# Best-loss model: expected-needs allocation performance

Choose the model with the lowest **validation loss separately for each split seed**, then evaluate its **saved allocations**. The selected model stays fixed across cohorts, algorithms, alpha replacements, and beta shrinkage values. This notebook reads existing files; it does not train models or rerun the allocation algorithms.

The default criterion is **validation cross-entropy of the need distribution q**. Set `BEST_LOSS = "validation_binary_log_loss"` to use the binary p-based validation loss currently used to construct the saved Rashomon sets. These criteria can choose different models. Training and test losses are never used to choose the best model. Exact ties are broken by the numeric model ID.

All allocation disparities use **true needs**. Comparisons include LDA-exp, LDA-tru, OPT, and available alpha/beta runs. The reference labelled “All saved models” includes the chosen model and uses the same model-level averaging as the existing results notebook. A separate paired table gives each cohort equal weight.

In [1]:
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/allocations.py").exists())


In [2]:
METHOD = "bootstrap"
MODEL_CLASS = "nn_multiclass"
PRED_METHODS = ["average"]
NEED_THRESHOLDS = [1]
SELECTION_RATES = [0.1]  # Add 0.25 and 0.5 to include the other saved selection rates.
SAMPLE_SIZE = 100
NEED_COL = "gagne_sum_t"
GROUP_COL = "race"
BEST_LOSS = "validation_cross_entropy"  # Or "validation_binary_log_loss".
INCLUDE_NEED_EXPERIMENTS = True

ALLOCATION_DIR = ROOT / "data/results/allocations"
METRICS = ["realized_utility", "expected_utility", "absolute_disparity", "disparity_ratio", "k_0", "k_1"]
CONFIG_KEYS = ["pred_variant", "tau", "selection_rate", "k", "n"]
RUN_KEYS = CONFIG_KEYS + ["experiment", "algorithm", "alpha", "beta"]
BEST_VIEW = "Best validation model"
ALL_VIEW = "All saved models"


In [3]:
def load_predictions_and_choose_models(pred_variant, tau, group_data):
    qualification = f"qualified_gagne_{tau}"
    name = f"test_{METHOD}_{MODEL_CLASS}_{pred_variant}_{qualification}.csv"
    predictions = pd.read_csv(ROOT / "data/predictions" / name).merge(
        group_data, on="person_id", how="left", validate="many_to_one"
    )
    model_columns = [column for column in predictions if column.startswith("m_")]
    validation_rows = predictions.loc[predictions["person_id"].eq(-1)]
    if not validation_rows.groupby("seed").size().eq(1).all():
        raise ValueError("Expected one binary validation-loss row per seed.")
    binary_losses = validation_rows.melt(
        id_vars="seed", value_vars=model_columns,
        var_name="model", value_name="validation_binary_log_loss",
    )
    losses = pd.read_csv(ROOT / "data/predictions/losses" / name)
    losses = losses.merge(binary_losses, on=["seed", "model"], validate="one_to_one", how="outer")
    if BEST_LOSS not in {"validation_cross_entropy", "validation_binary_log_loss"}:
        raise ValueError("BEST_LOSS must be a supported validation loss, not a training or test loss.")
    if BEST_LOSS not in losses or not np.isfinite(losses[BEST_LOSS]).all():
        raise ValueError(f"Missing or non-finite {BEST_LOSS} in the saved losses for {name}.")
    losses["model_number"] = losses["model"].str.removeprefix("m_").astype(int)
    losses["n_candidates"] = losses.groupby("seed")["model"].transform("size")
    best = (
        losses.sort_values(["seed", BEST_LOSS, "model_number"])
        .drop_duplicates("seed")
        .copy()
    )
    best["selection_loss_name"] = BEST_LOSS
    best["selection_loss"] = best[BEST_LOSS]
    best["pred_variant"], best["tau"] = pred_variant, tau
    real_predictions = predictions.loc[predictions["person_id"].ge(0)].copy()
    if real_predictions.duplicated(["seed", "person_id"]).any():
        raise ValueError("Duplicate people within a prediction split.")
    if real_predictions[GROUP_COL].isna().any():
        raise ValueError("Missing group labels for prediction rows.")
    if set(best["seed"]) != set(real_predictions["seed"]):
        raise ValueError("Prediction seeds and validation-loss seeds differ.")
    return real_predictions, best


def discover_saved_runs(tag):
    """Keep baseline and sweep endpoints separate, even when their settings coincide."""
    runs = []
    for algorithm, prefix in [("LDA-exp", "lda_exp"), ("LDA-tru", "lda_tru"), ("OPT", "opt")]:
        runs.append(dict(
            experiment="baseline", algorithm=algorithm,
            alpha=0.0 if algorithm == "LDA-exp" else np.nan,
            beta=1.0 if algorithm == "LDA-exp" else np.nan,
            people_path=ALLOCATION_DIR / f"people_{tag}.csv",
            allocation_path=ALLOCATION_DIR / f"{prefix}_{tag}.csv",
        ))
    if INCLUDE_NEED_EXPERIMENTS:
        for parameter in ["alpha", "beta"]:
            prefix = f"lda_exp_{tag}_{parameter}-"
            seen = set()
            for allocation_path in sorted(ALLOCATION_DIR.glob(f"{prefix}*.csv")):
                suffix = allocation_path.stem[len(prefix):]
                value = float(suffix)
                if not np.isfinite(value) or not 0 <= value <= 1 or value in seen:
                    raise ValueError(f"Invalid or repeated {parameter}: {allocation_path.name}")
                seen.add(value)
                people_path = ALLOCATION_DIR / f"people_{tag}_{parameter}-{suffix}.csv"
                if not people_path.exists():
                    print(f"Skipping unfinished saved run: {allocation_path.name}")
                    continue
                runs.append(dict(
                    experiment=parameter, algorithm="LDA-exp",
                    alpha=value if parameter == "alpha" else 0.0,
                    beta=value if parameter == "beta" else 1.0,
                    people_path=people_path, allocation_path=allocation_path,
                ))
    return runs


def evaluate_saved_run(predictions, chosen, run, config):
    people = pd.read_csv(run["people_path"], converters={"people": literal_eval})
    allocations = pd.read_csv(run["allocation_path"], converters={"selected": literal_eval, "models": literal_eval})
    if people.duplicated(["seed", "iteration"]).any():
        raise ValueError(f"Duplicate cohort rows: {run['people_path'].name}")
    best_by_seed = chosen.set_index("seed")["model"]
    membership = allocations[["seed", "iteration", "models"]].explode("models").rename(columns={"models": "model"})
    if membership.duplicated(["seed", "iteration", "model"]).any():
        raise ValueError(f"Repeated allocation for a model and cohort: {run['allocation_path'].name}")
    expected = people[["seed", "iteration"]].copy()
    expected["model"] = expected["seed"].map(best_by_seed)
    coverage = expected.merge(membership, on=["seed", "iteration", "model"], how="left", indicator=True)
    missing = coverage.loc[coverage["_merge"].eq("left_only"), ["seed", "iteration", "model"]]
    if not missing.empty:
        raise ValueError(
            f"The chosen best model has no saved allocation in {run['allocation_path'].name}: "
            f"{missing.to_dict('records')}. No fallback model has been substituted."
        )
    predictions_by_seed = {seed: frame.set_index("person_id") for seed, frame in predictions.groupby("seed")}
    allocations_by_cohort = {(seed, iteration): frame for (seed, iteration), frame in allocations.groupby(["seed", "iteration"])}
    rows, cohorts = [], []
    for cohort_info in people.itertuples(index=False):
        seed, iteration = cohort_info.seed, cohort_info.iteration
        if len(cohort_info.people) != SAMPLE_SIZE or len(set(cohort_info.people)) != SAMPLE_SIZE:
            raise ValueError("Saved cohort size does not match SAMPLE_SIZE.")
        cohort = predictions_by_seed[seed].loc[cohort_info.people]
        cohorts.append(dict(seed=seed, iteration=iteration, people=tuple(sorted(cohort_info.people))))
        for allocation in allocations_by_cohort[(seed, iteration)].itertuples(index=False):
            if len(allocation.selected) != config["k"] or len(set(allocation.selected)) != config["k"]:
                raise ValueError("Saved selection size does not match the selection rate.")
            selected = cohort.loc[allocation.selected]
            selected_0 = selected.loc[selected[GROUP_COL].eq(0), NEED_COL]
            selected_1 = selected.loc[selected[GROUP_COL].eq(1), NEED_COL]
            mean_0, mean_1 = selected_0.mean(), selected_1.mean()
            ratio = np.nan
            if pd.notna(mean_0) and pd.notna(mean_1) and mean_0 != 0 and mean_1 != 0:
                ratio = max(mean_1 / mean_0, mean_0 / mean_1)
            for model in allocation.models:
                rows.append({
                    **config,
                    **{key: run[key] for key in ["experiment", "algorithm", "alpha", "beta"]},
                    "seed": seed, "iteration": iteration, "model": model,
                    "is_best_model": model == best_by_seed.loc[seed],
                    "expected_utility": selected[model].mean(),
                    "realized_utility": selected["y"].mean(),
                    "absolute_disparity": abs(mean_1 - mean_0), "disparity_ratio": ratio,
                    "k_0": len(selected_0), "k_1": len(selected_1),
                })
    return pd.DataFrame(rows), pd.DataFrame(cohorts).sort_values(["seed", "iteration"]).reset_index(drop=True)


In [ ]:
group_data = pd.read_csv(ROOT / "data/obermeyer/obermeyer_data_cleaned.csv", usecols=["person_id", GROUP_COL])
result_frames, chosen_frames = [], []
for pred_variant in PRED_METHODS:
    for tau in NEED_THRESHOLDS:
        predictions, chosen = load_predictions_and_choose_models(pred_variant, tau, group_data)
        chosen_frames.append(chosen)
        for rate in SELECTION_RATES:
            config = dict(pred_variant=pred_variant, tau=tau, selection_rate=rate, k=int(rate * SAMPLE_SIZE), n=SAMPLE_SIZE)  # noqa: C408
            tag = f"{METHOD}_{MODEL_CLASS}_{pred_variant}_y-qualified_gagne_{tau}_rate-{rate}_size-{SAMPLE_SIZE}"
            reference_cohorts = None
            for run in discover_saved_runs(tag):
                results, cohorts = evaluate_saved_run(predictions, chosen, run, config)
                if reference_cohorts is None:
                    reference_cohorts = cohorts
                else:
                    pd.testing.assert_frame_equal(reference_cohorts, cohorts, check_dtype=False)
                result_frames.append(results)

df_model_results = pd.concat(result_frames, ignore_index=True)
best_model_results = df_model_results.loc[df_model_results["is_best_model"]].copy()
comparison_results = pd.concat([
    df_model_results.assign(model_selection=ALL_VIEW),
    best_model_results.assign(model_selection=BEST_VIEW),
], ignore_index=True)
best_models = pd.concat(chosen_frames, ignore_index=True)
print(f"Loaded {len(df_model_results):,} saved model-allocation evaluations; "
      f"{len(best_model_results):,} use the selected best model.")


Loaded 42,330 saved model-allocation evaluations; 425 use the selected best model.


## Which models were selected?

Model IDs identify separately fitted models within each seed. A best model is selected from all models with saved validation losses, then its presence in every requested allocation run is checked. Missing allocations raise an error rather than silently choosing another model. The binary validation loss is shown alongside the q-based loss so the selection criterion is explicit.

In [5]:
best_model_table = best_models[[
    "pred_variant", "tau", "seed", "model", "selection_loss_name", "selection_loss",
    "validation_cross_entropy", "validation_binary_log_loss", "test_cross_entropy", "n_candidates",
]].sort_values(["pred_variant", "tau", "seed"]).reset_index(drop=True)
best_model_table


,pred_variant,tau,seed,model,selection_loss_name,selection_loss,validation_cross_entropy,validation_binary_log_loss,test_cross_entropy,n_candidates
0,average,1,0,m_47,validation_cross_entropy,0.940110,0.940110,0.326374,0.935396,100
1,average,1,1,m_91,validation_cross_entropy,0.945488,0.945488,0.327935,0.939131,100
2,average,1,2,m_72,validation_cross_entropy,0.929616,0.929616,0.326949,0.928832,100
3,average,1,3,m_67,validation_cross_entropy,0.949572,0.949572,0.327918,0.931197,100
4,average,1,4,m_90,validation_cross_entropy,0.925259,0.925259,0.335448,0.915053,100


## Allocation performance: best model versus all saved models

The best-model rows have one allocation per cohort; their standard deviations describe variation across cohorts. The all-model rows include one allocation per model per cohort, matching the weighting in `03-results_exp_needs_multiclass.ipynb`. Shared selected sets are expanded to their constituent models. The counts below make the different numbers of observations explicit.

Absolute disparity is undefined when a selected group is empty; the ratio is also undefined when either group has zero mean need. As in the existing notebook, these values are omitted from the corresponding mean and standard deviation. `n_valid_disparity` reports how many allocations contribute to the disparity summary; for OPT this can be smaller than `n_allocations`.

Cross-model consistency and systemic-exclusion metrics are omitted here: with one model per cohort, pairwise consistency is trivially 1 and “never selected by any model” reduces to that model's ordinary nonselection. These would not measure model multiplicity in the best-model view.

In [6]:
summary_keys = RUN_KEYS + ["model_selection"]
performance_stats = comparison_results.groupby(summary_keys, dropna=False)[METRICS].agg(["mean", "std", "count"])
summary_counts = comparison_results.groupby(summary_keys, dropna=False).agg(
    n_allocations=("model", "size"), n_seeds=("seed", "nunique"),
)
summary_counts["n_cohorts"] = (
    comparison_results.drop_duplicates(summary_keys + ["seed", "iteration"])
    .groupby(summary_keys, dropna=False).size()
)
comparison_summary = summary_counts.copy()
comparison_summary["n_valid_disparity"] = performance_stats[("absolute_disparity", "count")]
for metric in METRICS:
    comparison_summary[metric] = performance_stats[metric].apply(
        lambda row: f"{row['mean']:.2f} ± {row['std']:.2f}", axis=1
    )
comparison_summary


n_allocations  \
pred_variant tau selection_rate k  n   experiment algorithm alpha beta model_selection                        
average      1   0.1            10 100 alpha      LDA-exp   0.00  1.00 All saved models                2490   
                                                                       Best validation model             25   
                                                            0.10  1.00 All saved models                2490   
                                                                       Best validation model             25   
                                                            0.20  1.00 All saved models                2490   
                                                                       Best validation model             25   
                                                            0.25  1.00 All saved models                2490   
                                                                       Best validation model             25   
                                                            0.50  1.00 All saved models                2490   
                                                                       Best validation model             25   
                                                            0.70  1.00 All saved models                2490   
                                                                       Best validation model             25   
                                                            0.75  1.00 All saved models                2490   
                                                                       Best validation model             25   
                                                            0.90  1.00 All saved models                2490   
                                                                       Best validation model             25   
                                                            1.00  1.00 All saved models                2490   
                                                                       Best validation model             25   
                                       baseline   LDA-exp   0.00  1.00 All saved models                2490   
                                                                       Best validation model             25   
                                                  LDA-tru   NaN   NaN  All saved models                2490   
                                                                       Best validation model             25   
                                                  OPT       NaN   NaN  All saved models                2490   
                                                                       Best validation model             25   
                                       beta       LDA-exp   0.00  0.00 All saved models                2490   
                                                                       Best validation model             25   
                                                                  0.25 All saved models                2490   
                                                                       Best validation model             25   
                                                                  0.50 All saved models                2490   
                                                                       Best validation model             25   
                                                                  0.75 All saved models                2490   
                                                                       Best validation model             25   
                                                                  1.00 All saved models                2490   
                                                                       Best validation model             25   

                                                                                              n_s

## Absolute disparity comparison

Each point is the mean absolute disparity for an experiment. Lighter horizontal error bars show ±1 standard deviation, with the lower bound clipped at zero. Blue points use the chosen best model; gray points use all saved models. Separate rows preserve alpha/beta endpoint runs even when they coincide with a baseline.

In [7]:
plot_stats = performance_stats["absolute_disparity"].reset_index()
best_loss_figures = []
for config, stats in plot_stats.groupby(CONFIG_KEYS, dropna=False, sort=True):
    pred_variant, tau, rate, k, n = config
    stats = stats.copy()
    stats["experiment_order"] = stats["experiment"].map({"baseline": 0, "alpha": 1, "beta": 2})
    stats["algorithm_order"] = stats["algorithm"].map({"LDA-exp": 0, "LDA-tru": 1, "OPT": 2})
    stats = stats.sort_values(["experiment_order", "algorithm_order", "alpha", "beta"])
    stats["run_label"] = stats.apply(
        lambda row: row["algorithm"] if row["experiment"] == "baseline" else
        f"LDA-exp · α={row['alpha']:g}" if row["experiment"] == "alpha" else
        f"LDA-exp · β={row['beta']:g}", axis=1,
    )
    labels = stats["run_label"].drop_duplicates().tolist()
    positions = {label: index for index, label in enumerate(labels)}
    fig = go.Figure()
    for view, color, lighter, offset in [
        (BEST_VIEW, "#0072B2", "rgba(0,114,178,0.30)", -0.13),
        (ALL_VIEW, "#777777", "rgba(119,119,119,0.25)", 0.13),
    ]:
        points = stats.loc[stats["model_selection"].eq(view)]
        fig.add_trace(go.Scatter(
            x=points["mean"].tolist(),
            y=(points["run_label"].map(positions) + offset).tolist(),
            customdata=points[["run_label", "std", "count"]].to_numpy().tolist(),
            mode="markers", name=view, marker=dict(color=color, size=9),
            error_x=dict(type="data", symmetric=False, array=points["std"].tolist(),
                         arrayminus=np.minimum(points["std"], points["mean"]).tolist(),
                         color=lighter, thickness=3, width=4),
            hovertemplate="%{customdata[0]}<br>Mean: %{x:.3f}<br>SD: %{customdata[1]:.3f}"
                          "<br>Valid allocations: %{customdata[2]}<extra>%{fullData.name}</extra>",
        ))
    fig.update_layout(
        template="plotly_white", height=max(420, 42 * len(labels) + 160),
        title=dict(text=f"Best validation model versus all saved models"
                        f"<br><sup>{pred_variant} · rate={rate:.0%} · k={int(k)} · n={int(n)}"
                        f" · threshold={tau:g} · {BEST_LOSS}</sup>", x=0.5, xanchor="center"),
        xaxis=dict(title="Absolute disparity evaluated on true needs", rangemode="tozero"),
        yaxis=dict(tickvals=list(positions.values()), ticktext=labels,
                   range=[len(labels) - 0.5, -0.5], showgrid=True),
        legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.10),
        margin=dict(l=150, r=35, t=95, b=90),
    )
    best_loss_figures.append(fig)
    fig.show()


Absolute disparity for the best validation model versus all saved models.

## Paired comparison on the same cohorts

For each cohort, subtract the all-model mean from the chosen model's result, then summarize these differences across cohorts. Every cohort with a defined difference has equal weight here. **Negative disparity differences favor the best model; positive utility differences favor it.** `n_valid_disparity_pairs` counts cohorts with a defined disparity in both views. Standard deviations show the variation in paired differences, not a confidence interval or significance test.

In [8]:
paired_keys = RUN_KEYS + ["seed", "iteration"]
cohort_all_model_means = df_model_results.groupby(paired_keys, dropna=False)[METRICS].mean().reset_index()
paired_results = best_model_results.merge(
    cohort_all_model_means, on=paired_keys, suffixes=("_best", "_all_mean"), validate="one_to_one"
)
delta_metrics = ["absolute_disparity", "realized_utility", "expected_utility"]
for metric in delta_metrics:
    paired_results[f"{metric}_difference"] = paired_results[f"{metric}_best"] - paired_results[f"{metric}_all_mean"]
paired_columns = [f"{metric}_difference" for metric in delta_metrics]
paired_stats = paired_results.groupby(RUN_KEYS, dropna=False)[paired_columns].agg(["mean", "std", "count"])
paired_summary = pd.DataFrame(index=paired_stats.index)
paired_summary["n_cohorts"] = paired_results.groupby(RUN_KEYS, dropna=False).size()
paired_summary["n_valid_disparity_pairs"] = paired_stats[("absolute_disparity_difference", "count")]
for metric in paired_columns:
    paired_summary[metric] = paired_stats[metric].apply(
        lambda row: f"{row['mean']:+.2f} ± {row['std']:.2f}", axis=1
    )
paired_summary


n_cohorts  \
pred_variant tau selection_rate k  n   experiment algorithm alpha beta              
average      1   0.1            10 100 alpha      LDA-exp   0.00  1.00         25   
                                                            0.10  1.00         25   
                                                            0.20  1.00         25   
                                                            0.25  1.00         25   
                                                            0.50  1.00         25   
                                                            0.70  1.00         25   
                                                            0.75  1.00         25   
                                                            0.90  1.00         25   
                                                            1.00  1.00         25   
                                       baseline   LDA-exp   0.00  1.00         25   
                                                  LDA-tru   NaN   NaN          25   
                                                  OPT       NaN   NaN          25   
                                       beta       LDA-exp   0.00  0.00         25   
                                                                  0.25         25   
                                                                  0.50         25   
                                                                  0.75         25   
                                                                  1.00         25   

                                                                        n_valid_disparity_pairs  \
pred_variant tau selection_rate k  n   experiment algorithm alpha beta                            
average      1   0.1            10 100 alpha      LDA-exp   0.00  1.00                       25   
                                                            0.10  1.00                       25   
                                                            0.20  1.00                       25   
                                                            0.25  1.00                       25   
                                                            0.50  1.00                       25   
                                                            0.70  1.00                       25   
                                                            0.75  1.00                       25   
                                                            0.90  1.00                       25   
                                                            1.00  1.00                       25   
                                       baseline   LDA-exp   0.00  1.00                       25   
                                                  LDA-tru   NaN   NaN                        25   
                                                  OPT       NaN   NaN                        18   
                                       beta       LDA-exp   0.00  0.00                       25   
                                                                  0.25                       25   
                                                                  0.50                       25   
                                                                  0.75                       25   
                                                                  1.00                       25   

                                                                       absolute_disparity_difference  \
pred_variant tau selection_rate k  n   experiment algorithm alpha beta                                 
average      1   0.1            10 100 alpha      LDA-exp   0.00  1.00                  +0.08 ± 0.75   
                                                            0.10  1.00                  -0.01 ± 0.66   
                                                            0.20  1.00                  -0.05 ± 0.53   
                                                      

## Original LDA-exp performance by seed

Inspect whether the best model's performance is consistent across split seeds. These rows use the original LDA-exp allocations, before alpha replacement or beta shrinkage. `best_model_results` contains the individual cohort results for all experiments.

In [9]:
baseline_best = best_model_results.loc[
    best_model_results["experiment"].eq("baseline") & best_model_results["algorithm"].eq("LDA-exp")
]
seed_keys = CONFIG_KEYS + ["seed", "model"]
by_seed_stats = baseline_best.groupby(seed_keys, dropna=False)[METRICS].agg(["mean", "std"])
best_baseline_by_seed = pd.DataFrame(index=by_seed_stats.index)
best_baseline_by_seed["n_cohorts"] = baseline_best.groupby(seed_keys, dropna=False).size()
for metric in METRICS:
    best_baseline_by_seed[metric] = by_seed_stats[metric].apply(
        lambda row: f"{row['mean']:.2f} ± {row['std']:.2f}", axis=1
    )
best_baseline_by_seed


n_cohorts realized_utility  \
pred_variant tau selection_rate k  n   seed model                               
average      1   0.1            10 100 0    m_47           5      0.98 ± 0.04   
                                       1    m_91           5      0.94 ± 0.09   
                                       2    m_72           5      0.94 ± 0.09   
                                       3    m_67           5      0.98 ± 0.04   
                                       4    m_90           5      0.92 ± 0.08   

                                                  expected_utility  \
pred_variant tau selection_rate k  n   seed model                    
average      1   0.1            10 100 0    m_47       0.98 ± 0.00   
                                       1    m_91       0.95 ± 0.05   
                                       2    m_72       0.98 ± 0.01   
                                       3    m_67       0.99 ± 0.00   
                                       4    m_90       0.97 ± 0.04   

                                                  absolute_disparity  \
pred_variant tau selection_rate k  n   seed model                      
average      1   0.1            10 100 0    m_47         1.07 ± 0.72   
                                       1    m_91         2.52 ± 1.24   
                                       2    m_72         2.46 ± 1.86   
                                       3    m_67         0.82 ± 0.56   
                                       4    m_90         1.79 ± 1.20   

                                                  disparity_ratio  \
pred_variant tau selection_rate k  n   seed model                   
average      1   0.1            10 100 0    m_47      1.36 ± 0.43   
                                       1    m_91      2.41 ± 1.46   
                                       2    m_72      1.78 ± 0.53   
                                       3    m_67      1.24 ± 0.18   
                                       4    m_90      1.35 ± 0.15   

                                                           k_0          k_1  
pred_variant tau selection_rate k  n   seed model                            
average      1   0.1            10 100 0    m_47   3.20 ± 2.95  6.80 ± 2.95  
                                       1    m_91   1.60 ± 0.89  8.40 ± 0.89  
                                       2    m_72   6.80 ± 2.39  3.20 ± 2.39  
                                       3    m_67   5.00 ± 1.87  5.00 ± 1.87  
                                       4    m_90   3.80 ± 2.68  6.20 ± 2.68